In [1]:
# --- 1. IMPORTS ---

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shutil
import glob
from spark_session import get_spark_session

print("✓ Imports completed.")

✓ Imports completed.


In [2]:
# --- 2. CONFIGURATION PARAMETERS ---

# Host paths (for Driver to read files)
HOST_DATA_PATH = "./../../data" 
# Container paths (inside Docker as in docker-compose.yml)
CONTAINER_DATA_PATH = "/opt/spark/data"

# Output paths
OUTPUT_HOST_PATH = os.path.join(HOST_DATA_PATH, "processed")
OUTPUT_CONTAINER_PATH = os.path.join(CONTAINER_DATA_PATH, "processed")

# R2 bucket prefix (for future use)
UPLOAD_PREFIX = 'processed/'

# Slice configuration
FROM_N_SLICES = 3
TO_N_SLICES = 6

# Limit number of images to process (for testing)
IMAGE_LIMIT = 1

print(f"Configuration:")
print(f"  - Processing slices from {FROM_N_SLICES} to {TO_N_SLICES}")
print(f"  - Image limit: {IMAGE_LIMIT}")
print(f"  - Output path: {OUTPUT_HOST_PATH}")

Configuration:
  - Processing slices from 3 to 6
  - Image limit: 1
  - Output path: ./../../data/processed


In [3]:
# --- 3. INICIALIZAR SESIÓN DE SPARK ---
print("Iniciando SparkSession en modo distribuido...")
spark = get_spark_session("TreeRingSlicing (Distributed)")

sc = spark.sparkContext
print("\n--- SparkSession Iniciada --- ✅")
print(f"Spark Version: {sc.version}")
print(f"Master: {sc.master}")
print(f"UI Web: {sc.uiWebUrl}")

Iniciando SparkSession en modo distribuido...
Attempting to connect to master at: spark://localhost:7077
Will announce driver host IP as: 172.26.0.1
Check: IP 172.26.0.1 resolves locally.


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/09 23:23:15 WARN Utils: Your hostname, archlinux, resolves to a loopback address: 127.0.0.1; using 192.168.1.5 instead (on interface wlp0s20f3)
25/11/09 23:23:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/09 23:23:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable



--- Connection Successful! --- ✅
SparkSession object: <pyspark.sql.session.SparkSession object at 0x7f193d93a440>
SparkContext object: <SparkContext master=spark://localhost:7077 appName=TreeRingSlicing (Distributed)>
Spark version in use: 4.0.1

--- SparkSession Iniciada --- ✅
Spark Version: 4.0.1
Master: spark://localhost:7077
UI Web: http://172.26.0.1:4040


In [4]:
# --- 4. HELPER FUNCTIONS ---

def load_pith_locations(metadata_path):
    """
    Load pith center locations from CSV into a dictionary for fast lookup.
    """
    try:
        csv_path = os.path.join(metadata_path, 'pith_location.csv')
        df = pd.read_csv(csv_path)
        # Create dictionary: {'F10b': (cx, cy)}
        pith_dict = {row['Image']: (row['cx'], row['cy']) for index, row in df.iterrows()}
        
        print(f"  ✓ Loaded {len(pith_dict)} pith locations from CSV")
        return pith_dict
        
    except FileNotFoundError:
        print(f"  ✗ Error: CSV file not found at {csv_path}")
        return {}
    except Exception as e:
        print(f"  ✗ Error loading CSV: {e}")
        return {}


def load_ring_counts(metadata_path):
    """
    Load ring counts from CSV into a dictionary for fast lookup.
    """
    try:
        csv_path = os.path.join(metadata_path, 'ring_counts.csv')
        df = pd.read_csv(csv_path)
        # Create dictionary: {'F10b': 23}
        ring_dict = {row['image_code']: int(row['rings_count']) for index, row in df.iterrows()}
        
        print(f"  ✓ Loaded {len(ring_dict)} ring counts from CSV")
        return ring_dict
        
    except FileNotFoundError:
        print(f"  ✗ Error: CSV file not found at {csv_path}")
        return {}
    except Exception as e:
        print(f"  ✗ Error loading CSV: {e}")
        return {}


def get_image_files(raw_path, limit=None):
    """
    Scan the 'raw' folder and return a list of image filenames.
    """
    try:
        all_files = [f for f in os.listdir(raw_path) if f.lower().endswith('.png')]
        all_files.sort()
        
        if limit is not None:
            print(f"  ✓ Found {len(all_files)} images. Processing first {limit}")
            return all_files[:limit]
        else:
            print(f"  ✓ Found and will process {len(all_files)} images")
            return all_files
            
    except FileNotFoundError:
        print(f"  ✗ Error: 'raw' directory not found at {raw_path}")
        return []

In [5]:
# --- 5. WORKER PROCESSING LOGIC ---

def process_image_multi_slices(image_name, base_data_path, output_base_path, 
                                from_n_slices, to_n_slices,
                                pith_broadcast, ring_counts_broadcast):
    """
    Process a single image and generate slices for all N values (from_n_slices to to_n_slices).
    Each slice is cropped to its bounding box and saved with ring_count metadata.
    
    This function will run in parallel on Spark workers.
    """
    try:
        # --------------------------------------------------
        # 1. Setup paths and load image
        # --------------------------------------------------
        image_path = os.path.join(base_data_path, 'raw', image_name)
        image_name_no_ext = os.path.splitext(image_name)[0]
        
        # Get broadcast dictionaries
        pith_map = pith_broadcast.value
        ring_counts_map = ring_counts_broadcast.value
        
        # Load image
        img = cv2.imread(image_path)
        if img is None:
            return (image_name, f"ERROR: Could not read image")
            
        h, w, _ = img.shape
        
        # --------------------------------------------------
        # 2. Get metadata for this image
        # --------------------------------------------------
        if image_name_no_ext not in pith_map:
            return (image_name, f"ERROR: Pith center not found for {image_name_no_ext}")
        
        if image_name_no_ext not in ring_counts_map:
            return (image_name, f"ERROR: Ring count not found for {image_name_no_ext}")
            
        # Get pith center (cy, cx format from CSV)
        (cy, cx) = pith_map[image_name_no_ext]
        
        # Get ring count (all slices inherit this value)
        ring_count = ring_counts_map[image_name_no_ext]
        
        # Calculate maximum radius to cover entire image from center
        radius = int(np.sqrt(max(cx, w - cx)**2 + max(cy, h - cy)**2)) + 1
        
        # --------------------------------------------------
        # 3. Generate slices for each N value
        # --------------------------------------------------
        total_slices_generated = 0
        total_original_pixels = 0
        total_cropped_pixels = 0
        
        for n_slices in range(from_n_slices, to_n_slices + 1):
            # Create output directory for this N and image
            output_dir = os.path.join(output_base_path, str(n_slices), image_name_no_ext)
            os.makedirs(output_dir, exist_ok=True)
            
            angle_per_slice = 360.0 / n_slices
            
            for i in range(n_slices):
                # Calculate angles
                start_angle = (i * angle_per_slice) - 90
                end_angle = ((i + 1) * angle_per_slice) - 90
                
                # Create mask
                mask = np.zeros((h, w), dtype=np.uint8)
                cv2.ellipse(
                    mask,
                    center=(int(cx), int(cy)),
                    axes=(int(radius), int(radius)),
                    angle=0,
                    startAngle=start_angle,
                    endAngle=end_angle,
                    color=255,
                    thickness=-1
                )
                
                # Apply mask
                result = cv2.bitwise_and(img, img, mask=mask)
                
                # Crop to bounding box
                contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                
                if contours:
                    x, y, bbox_w, bbox_h = cv2.boundingRect(contours[0])
                    cropped_result = result[y:y+bbox_h, x:x+bbox_w]
                    
                    # Track pixel savings
                    total_original_pixels += (w * h)
                    total_cropped_pixels += (bbox_w * bbox_h)
                else:
                    cropped_result = result
                
                # Save with metadata in filename (for local testing)
                # Format: imagename_slicenumber.png
                output_filename = f"{image_name_no_ext}_{i+1}.png"
                output_path = os.path.join(output_dir, output_filename)
                cv2.imwrite(output_path, cropped_result)
                
                total_slices_generated += 1
        
        # Calculate statistics
        space_savings = 0
        if total_original_pixels > 0:
            space_savings = ((total_original_pixels - total_cropped_pixels) / total_original_pixels) * 100
        
        n_range = to_n_slices - from_n_slices + 1
        return (image_name, 
                f"SUCCESS: {total_slices_generated} slices generated "
                f"({n_range} N values × varying slices/N) "
                f"| Ring count: {ring_count} "
                f"| Space saved: {space_savings:.1f}%")
        
    except Exception as e:
        return (image_name, f"ERROR: {str(e)}")

In [8]:
# --- 6. SPARK PIPELINE EXECUTION ---

print("=" * 80)
print("MULTI-SLICE IMAGE PROCESSING PIPELINE (LOCAL VALIDATION)")
print("=" * 80)

# ------------------------------------------------------------------------------
# STEP 0: Prepare output directory with proper permissions
# ------------------------------------------------------------------------------
print("\n[0/6] Preparing output directory...")

os.makedirs(OUTPUT_HOST_PATH, exist_ok=True)

# Clean previous outputs for all N_SLICES we'll generate
print(f"  → Cleaning output directory: {OUTPUT_HOST_PATH}")
if os.path.exists(OUTPUT_HOST_PATH):
    # Only clean if it exists and has content
    for n in range(FROM_N_SLICES, TO_N_SLICES + 1):
        n_dir = os.path.join(OUTPUT_HOST_PATH, str(n))
        if os.path.exists(n_dir):
            shutil.rmtree(n_dir)

# Set proper permissions for Docker containers
os.chmod(OUTPUT_HOST_PATH, 0o777)
print(f"  ✓ Output directory ready: {OUTPUT_HOST_PATH}")
print(f"  ✓ Will create subdirectories for N={FROM_N_SLICES} to N={TO_N_SLICES}")

# ------------------------------------------------------------------------------
# STEP 1: Load and broadcast metadata (Driver-side)
# ------------------------------------------------------------------------------
print("\n[1/6] Loading metadata (Driver-side)...")

# Load pith locations
pith_dict = load_pith_locations(os.path.join(HOST_DATA_PATH, 'metadata'))
if not pith_dict:
    raise ValueError("Failed to load pith data. Aborting.")

# Load ring counts
ring_counts_dict = load_ring_counts(os.path.join(HOST_DATA_PATH, 'metadata'))
if not ring_counts_dict:
    raise ValueError("Failed to load ring counts data. Aborting.")

# Broadcast to workers
pith_broadcast = sc.broadcast(pith_dict)
ring_counts_broadcast = sc.broadcast(ring_counts_dict)

print(f"  ✓ Broadcast {len(pith_dict)} pith locations")
print(f"  ✓ Broadcast {len(ring_counts_dict)} ring counts")
print(f"  Example: F02a has {ring_counts_dict.get('F02a', 'N/A')} rings")

# ------------------------------------------------------------------------------
# STEP 2: Get list of images to process (Driver-side)
# ------------------------------------------------------------------------------
print("\n[2/6] Scanning for image files (Driver-side)...")

image_files = get_image_files(os.path.join(HOST_DATA_PATH, 'raw'), limit=IMAGE_LIMIT)
if not image_files:
    raise ValueError("No images found to process. Aborting.")

print(f"  ✓ Found {len(image_files)} image(s) to process")

# ------------------------------------------------------------------------------
# STEP 3: Calculate expected output
# ------------------------------------------------------------------------------
print("\n[3/6] Calculating expected output...")

total_n_values = TO_N_SLICES - FROM_N_SLICES + 1
total_slices_per_image = sum(range(FROM_N_SLICES, TO_N_SLICES + 1))
total_files_expected = len(image_files) * total_slices_per_image

print(f"  → N values to process: {total_n_values} (from {FROM_N_SLICES} to {TO_N_SLICES})")
print(f"  → Slices per image: {total_slices_per_image}")
print(f"  → Total files expected: {total_files_expected}")

# ------------------------------------------------------------------------------
# STEP 4: Create RDD and define worker function
# ------------------------------------------------------------------------------
print("\n[4/6] Creating RDD for parallel processing...")

image_rdd = sc.parallelize(image_files)
print(f"  ✓ RDD created with {image_rdd.getNumPartitions()} partition(s)")


def run_multi_slicing(image_name):
    """
    Wrapper function that will run on each Spark worker.
    Passes container paths to the processing function.
    """
    return process_image_multi_slices(
        image_name,
        CONTAINER_DATA_PATH,
        OUTPUT_CONTAINER_PATH,
        FROM_N_SLICES,
        TO_N_SLICES,
        pith_broadcast,
        ring_counts_broadcast
    )

# ------------------------------------------------------------------------------
# STEP 5: Execute processing
# ------------------------------------------------------------------------------
print("\n[5/6] Executing distributed multi-slice generation...")
print(f"  ⏳ Processing {len(image_files)} image(s) × {total_n_values} N values...")

results = image_rdd.map(run_multi_slicing).collect()

print(f"  ✓ Processing completed!")

# ------------------------------------------------------------------------------
# STEP 6: Display execution results
# ------------------------------------------------------------------------------
print("\n[6/6] Execution Summary:")
print("=" * 80)

success_count = 0
error_count = 0

for image_name, status in results:
    status_icon = "✓" if "SUCCESS" in status else "✗"
    print(f"  {status_icon} {image_name}")
    print(f"     {status}")
    
    if "SUCCESS" in status:
        success_count += 1
    else:
        error_count += 1

print("=" * 80)
print(f"\n📊 Final Statistics:")
print(f"  ✓ Successful: {success_count}")
print(f"  ✗ Errors: {error_count}")
print(f"  📁 Total files generated: {total_files_expected}")
print(f"  💾 Output location: {OUTPUT_HOST_PATH}")
print("\n✅ Pipeline execution completed!")
print("=" * 80)

MULTI-SLICE IMAGE PROCESSING PIPELINE (LOCAL VALIDATION)

[0/6] Preparing output directory...
  → Cleaning output directory: ./../../data/processed
  ✓ Output directory ready: ./../../data/processed
  ✓ Will create subdirectories for N=3 to N=6

[1/6] Loading metadata (Driver-side)...
  ✓ Loaded 64 pith locations from CSV
  ✓ Loaded 64 ring counts from CSV
  ✓ Broadcast 64 pith locations
  ✓ Broadcast 64 ring counts
  Example: F02a has 23 rings

[2/6] Scanning for image files (Driver-side)...
  ✓ Found 64 images. Processing first 1
  ✓ Found 1 image(s) to process

[3/6] Calculating expected output...
  → N values to process: 4 (from 3 to 6)
  → Slices per image: 18
  → Total files expected: 18

[4/6] Creating RDD for parallel processing...
  ✓ RDD created with 8 partition(s)

[5/6] Executing distributed multi-slice generation...
  ⏳ Processing 1 image(s) × 4 N values...


  ✓ Processing completed!

[6/6] Execution Summary:
  ✓ F02a.png
     SUCCESS: 18 slices generated (4 N values × varying slices/N) | Ring count: 23 | Space saved: 70.0%

📊 Final Statistics:
  ✓ Successful: 1
  ✗ Errors: 0
  📁 Total files generated: 18
  💾 Output location: ./../../data/processed

✅ Pipeline execution completed!


In [9]:
# --- 7. VERIFICATION: Check generated directory structure ---

print("\n" + "=" * 80)
print("VERIFICATION: Checking generated directory structure")
print("=" * 80)

# Check what was generated
for n_slices in range(FROM_N_SLICES, TO_N_SLICES + 1):
    n_dir = os.path.join(OUTPUT_HOST_PATH, str(n_slices))
    
    if os.path.exists(n_dir):
        # Count subdirectories (one per image)
        subdirs = [d for d in os.listdir(n_dir) if os.path.isdir(os.path.join(n_dir, d))]
        
        # Count total files
        total_files = 0
        for subdir in subdirs:
            subdir_path = os.path.join(n_dir, subdir)
            files = [f for f in os.listdir(subdir_path) if f.endswith('.png')]
            total_files += len(files)
        
        print(f"✓ N={n_slices:2d}: {len(subdirs)} image folder(s), {total_files} slice file(s)")
    else:
        print(f"✗ N={n_slices:2d}: Directory not found")

print("=" * 80)

# Show example structure for first N value
print(f"\nExample: Structure for N={FROM_N_SLICES}")
example_n_dir = os.path.join(OUTPUT_HOST_PATH, str(FROM_N_SLICES))
if os.path.exists(example_n_dir):
    for image_dir in sorted(os.listdir(example_n_dir))[:3]:  # Show first 3 images
        image_dir_path = os.path.join(example_n_dir, image_dir)
        if os.path.isdir(image_dir_path):
            files = sorted([f for f in os.listdir(image_dir_path) if f.endswith('.png')])
            print(f"\n  📁 {FROM_N_SLICES}/{image_dir}/")
            for f in files[:5]:  # Show first 5 files
                file_path = os.path.join(image_dir_path, f)
                file_size = os.path.getsize(file_path) / 1024  # KB
                print(f"     - {f} ({file_size:.1f} KB)")
            if len(files) > 5:
                print(f"     ... and {len(files) - 5} more files")
else:
    print(f"  Directory not found: {example_n_dir}")

print("\n" + "=" * 80)


VERIFICATION: Checking generated directory structure
✓ N= 3: 1 image folder(s), 3 slice file(s)
✓ N= 4: 1 image folder(s), 4 slice file(s)
✓ N= 5: 1 image folder(s), 5 slice file(s)
✓ N= 6: 1 image folder(s), 6 slice file(s)

Example: Structure for N=3

  📁 3/F02a/
     - F02a_1.png (2528.4 KB)
     - F02a_2.png (2973.2 KB)
     - F02a_3.png (3193.7 KB)



In [ ]:
# --- 8. STOP SPARK SESSION ---

print("\n🛑 Stopping SparkSession...")
spark.stop()
print("✓ Session stopped.")